In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('../../'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import seaborn as sns
import matplotlib.pyplot as plt
from machines.predictions.svr_prediction import SVR_Prediction
from machines.enums import SvrKernelEnum
from preprocesses.preprocess import Preprocess
from preprocesses.enums import TransformEnum, TypeFileEnum, StandardScaleEnum
from metrics.error_metric import ErrorMetric
from metrics.enums import MetricEnum, PlotLegends
import numpy as np

/tmp/ipykernel_65823/70858012.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Read the file and prepare data for SVR

In [4]:
file_csv = '../../../data/phen/original/obregon_phenotypic_original_151617.csv'
df = pd.read_csv(file_csv, header=0)
df.head()

,TGW,HI,BM,NDVI_VG1,NDVI_VG2,NDVI_VG3,NDVI_VG4,NDVI_VG,NDVI_GF1,NDVI_GF2,...,NDVI_UAV_1,NDVI_UAV_2,NDVI_UAV_3,NDVI_UAV_4,NDVI_UAV_5,CT_UAV_1,CT_UAV_2,CT_UAV_3,CT_UAV_4,YLD
0,40.510542,0.493706,1252.841830,0.171286,0.176257,0.375381,0.757965,0.444839,0.776698,0.744168,...,0.499727,0.458816,0.659201,0.754407,0.873936,25.451605,24.184386,24.083716,31.700124,617.4453
1,37.094174,0.492329,1403.919849,0.170764,0.214652,0.409778,0.782313,0.469060,0.770800,0.720762,...,0.486031,0.499782,0.693321,0.777470,0.878137,25.482403,24.005464,23.580308,31.270182,689.6464
2,44.201750,0.498393,1376.809117,0.156686,0.169272,0.334773,0.738150,0.417022,0.751064,0.667258,...,0.462325,0.431861,0.610026,0.708450,0.875478,25.643851,24.291071,24.444127,32.008681,687.2559
3,32.876596,0.516203,1234.698326,0.165603,0.190059,0.332481,0.675919,0.406757,0.728636,0.643564,...,0.413157,0.405293,0.560463,0.652515,0.844511,26.120276,24.171371,25.492758,34.282378,636.4969
4,41.374175,0.481962,1279.299001,0.187345,0.221703,0.429379,0.778879,0.472912,0.755801,0.731699,...,0.492748,0.494245,0.653331,0.738558,0.882129,25.619695,23.807970,23.970900,31.579817,616.5022


In [5]:
# Get the head of varaibles

print(list[df.columns])
x_cols = ['TGW', 'HI', 'BM', 'NDVI_VG1', 'NDVI_VG2', 'NDVI_VG3', 'NDVI_VG4',
       'NDVI_VG', 'NDVI_GF1', 'NDVI_GF2', 'NDVI_GF3', 'NDVI_GF4', 'NDVI_GF',
       'NDVI_UAV_1', 'NDVI_UAV_2', 'NDVI_UAV_3', 'NDVI_UAV_4', 'NDVI_UAV_5',
       'CT_UAV_1', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4']
yield_col = 'YLD'
print(x_cols)

list[Index(['TGW', 'HI', 'BM', 'NDVI_VG1', 'NDVI_VG2', 'NDVI_VG3', 'NDVI_VG4',
       'NDVI_VG', 'NDVI_GF1', 'NDVI_GF2', 'NDVI_GF3', 'NDVI_GF4', 'NDVI_GF',
       'NDVI_UAV_1', 'NDVI_UAV_2', 'NDVI_UAV_3', 'NDVI_UAV_4', 'NDVI_UAV_5',
       'CT_UAV_1', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4', 'YLD'],
      dtype='object')]
['TGW', 'HI', 'BM', 'NDVI_VG1', 'NDVI_VG2', 'NDVI_VG3', 'NDVI_VG4', 'NDVI_VG', 'NDVI_GF1', 'NDVI_GF2', 'NDVI_GF3', 'NDVI_GF4', 'NDVI_GF', 'NDVI_UAV_1', 'NDVI_UAV_2', 'NDVI_UAV_3', 'NDVI_UAV_4', 'NDVI_UAV_5', 'CT_UAV_1', 'CT_UAV_2', 'CT_UAV_3', 'CT_UAV_4']


In [6]:
ss = StandardScaler()
si = SimpleImputer( strategy='mean')
df[x_cols] = ss.fit_transform(df[x_cols])
df[x_cols] = si.fit_transform(df[x_cols])
df.head()



,TGW,HI,BM,NDVI_VG1,NDVI_VG2,NDVI_VG3,NDVI_VG4,NDVI_VG,NDVI_GF1,NDVI_GF2,...,NDVI_UAV_1,NDVI_UAV_2,NDVI_UAV_3,NDVI_UAV_4,NDVI_UAV_5,CT_UAV_1,CT_UAV_2,CT_UAV_3,CT_UAV_4,YLD
0,-0.110673,0.681610,-0.282279,-0.788747,-1.096127,-1.191792,0.408816,-1.155893,0.952816,1.065848,...,-0.820190,-1.116770,-0.823039,0.316080,0.348593,-1.063136,-0.937620,-0.912799,1.055163,617.4453
1,-0.820539,0.646893,0.537439,-0.792272,-0.961421,-1.017695,0.786018,-0.941195,0.884020,0.623312,...,-0.937790,-0.830913,0.059061,0.995764,0.552691,-1.043528,-0.993309,-1.053877,0.896796,689.6464
2,0.656301,0.799781,0.390342,-0.887363,-1.120633,-1.397326,0.101840,-1.402465,0.653816,-0.388284,...,-1.141340,-1.304860,-2.094353,-1.038305,0.423508,-0.940741,-0.904414,-0.811794,1.168818,687.2559
3,-1.696885,1.248814,-0.380722,-0.827133,-1.047704,-1.408927,-0.862250,-1.493456,0.392211,-0.836265,...,-1.563518,-1.490249,-3.375699,-2.686750,-1.080967,-0.637419,-0.941671,-0.517918,2.006322,636.4969
4,0.068776,0.385516,-0.138728,-0.680275,-0.936683,-0.918486,0.732818,-0.907051,0.709069,0.830098,...,-0.880115,-0.869550,-0.974795,-0.151001,0.746635,-0.956120,-1.054778,-0.944415,1.010849,616.5022


In [30]:
pca_1 = PCA(n_components=1 , random_state=42)
df_pca_1 = pca_1.fit_transform(df[x_cols])

# df_pca_2[:4]
len(pca_1.components_)
for i in range(len(x_cols)):
    print(f"{x_cols[i]}= {pca_1.components_[0][i]}")

TGW= 0.04306046872150792
HI= 0.01313688122021005
BM= 0.014181589996306391
NDVI_VG1= 0.203593523916693
NDVI_VG2= 0.2543450074257772
NDVI_VG3= 0.25096974963333946
NDVI_VG4= -0.21680893496760428
NDVI_VG= 0.2458788371240996
NDVI_GF1= -0.24331536749341265
NDVI_GF2= -0.17517163315479772
NDVI_GF3= -0.2274516472854885
NDVI_GF4= -0.25216759414052187
NDVI_GF= -0.24994240173195842
NDVI_UAV_1= 0.24903246334193557
NDVI_UAV_2= 0.25029970918600675
NDVI_UAV_3= 0.14639218757288897
NDVI_UAV_4= -0.13754581572412358
NDVI_UAV_5= -0.1933679218772529
CT_UAV_1= 0.2489626476246955
CT_UAV_2= 0.2547903789697979
CT_UAV_3= 0.25450544373584205
CT_UAV_4= -0.24376073290173247
